# 04 — Search tuning playground

The hybrid search pipeline has several levers you can pull at call time
(`rrf_weights`, `context_window`) and several more at construction time
(`mmr_config`, `decay_config`, `reranker`, `importance_config`). This
notebook focuses on the call-time levers because they are the ones you
will reach for most often, and shows how the same query reshuffles under
each setting.

The config-time levers are covered in the final cell with a recipe for
spinning up a second `Components` instance with your custom config.

## Prerequisites

- **memtomem** installed:
  ```bash
  # From PyPI
  uv pip install "memtomem[ollama]" jupyter ipykernel
  # Or from a source checkout
  uv pip install -e "packages/memtomem[all]" jupyter ipykernel
  ```
- Ollama running with `nomic-embed-text`.

In [1]:
# Verify Ollama is reachable before doing anything else.
import httpx

try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=2)
    r.raise_for_status()
except Exception as e:
    raise RuntimeError(
        "Ollama is not reachable at http://localhost:11434.\n"
        "Start it and pull the default embedding model:\n"
        "  ollama serve\n"
        "  ollama pull nomic-embed-text\n"
        "then re-run this cell."
    ) from e

print("Ollama is up.")

Ollama is up.


## Step 1 — Set up components and seed a small corpus

In [2]:
import json
import os
import tempfile
from pathlib import Path

from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import (
    Components,
    create_components,
    close_components,
)

# 1. Create an isolated temp directory so nothing lands in ~/.memtomem.
tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

# 2. Override config via environment variables. Note the double underscore —
#    pydantic-settings uses "__" as the nesting separator for section fields.
os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "ollama"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "nomic-embed-text"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "768"

# 3. Apply the same fields directly on the config object, and temporarily
#    disable load_config_overrides() so the real ~/.memtomem/config.json
#    cannot leak into this notebook. This mirrors the pattern used in
#    packages/memtomem/tests/conftest.py.
config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

import memtomem.config as _cfg

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None
try:
    comp: Components = await create_components(config)
finally:
    _cfg.load_config_overrides = _orig_loader

print(f"memtomem components ready. db={db_path}")

memtomem components ready. db=/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmpbdclmbrp/memory.db


In [3]:
corpus = {
    "rollback.md": """# Rollback drills

Always practice rolling back a deployment in staging before you actually
need to do it in production. A rollback you have never tested is not a
rollback, it is a wish.
""",
    "migrations.md": """# Database migrations

Run migrations in a dry-run mode first. Dry runs exercise the migration
path without touching production rows, which makes it safer to ship
schema changes alongside application code.
""",
    "monitoring.md": """# Deployment monitoring

Watch the error rate and p99 latency dashboards for fifteen minutes after
every deploy. If either metric regresses, roll back first and diagnose
later.
""",
    "playbook.md": """# On-call deploy playbook

When a deploy fails, follow the deploy playbook: page the on-call, start
the rollback, write a short post-incident note, and schedule a review.
""",
}

for name, content in corpus.items():
    (notes_dir / name).write_text(content, encoding="utf-8")

await comp.index_engine.index_path(notes_dir)

print(f"indexed {len(corpus)} files")

[04/11/26 11:06:22] INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=534639;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=12988;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=324981;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=149455;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=487553;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=408594;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=389287;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=333719;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

indexed 4 files


## Helper: pretty-print a result set

In [4]:
def pretty(title, results):
    print(f"--- {title} ---")
    for r in results:
        name = Path(r.chunk.metadata.source_file).name
        snippet = r.chunk.content[:80].replace("\n", " ")
        print(f"  [{r.rank}] {name:14s} score={r.score:.3f} via={r.source:<5} — {snippet}...")
    print()

## Variation 1 — Baseline (balanced BM25 + dense)

This is what the default `rrf_weights=[1.0, 1.0]` gives you.

In [5]:
query = "rollback after failed deploy"

results, stats = await comp.search_pipeline.search(query=query, top_k=4)
pretty(f"baseline (rrf=[1.0, 1.0], fused={stats.fused_total})", results)

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=433219;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=56704;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

--- baseline (rrf=[1.0, 1.0], fused=4) ---
  [1] monitoring.md  score=0.032 via=fused — Watch the error rate and p99 latency dashboards for fifteen minutes after every ...
  [2] playbook.md    score=0.032 via=fused — When a deploy fails, follow the deploy playbook: page the on-call, start the rol...
  [3] rollback.md    score=0.032 via=fused — Always practice rolling back a deployment in staging before you actually need to...
  [4] migrations.md  score=0.016 via=dense — Run migrations in a dry-run mode first. Dry runs exercise the migration path wit...



## Variation 2 — BM25-only

Setting the dense weight to zero disables the semantic side of the fusion
and lets you see what keyword matching alone is doing. Expect sharper
matches on exact terms and worse performance on paraphrases.

In [6]:
results, stats = await comp.search_pipeline.search(
    query=query,
    rrf_weights=[1.0, 0.0],
    top_k=4,
)
pretty(f"BM25 only (bm25={stats.bm25_candidates} candidates)", results)

--- BM25 only (bm25=3 candidates) ---
  [1] monitoring.md  score=0.032 via=fused — Watch the error rate and p99 latency dashboards for fifteen minutes after every ...
  [2] playbook.md    score=0.032 via=fused — When a deploy fails, follow the deploy playbook: page the on-call, start the rol...
  [3] rollback.md    score=0.032 via=fused — Always practice rolling back a deployment in staging before you actually need to...
  [4] migrations.md  score=0.016 via=dense — Run migrations in a dry-run mode first. Dry runs exercise the migration path wit...



## Variation 3 — Dense-only

The mirror image: only the vector retriever gets to vote.

In [7]:
results, stats = await comp.search_pipeline.search(
    query=query,
    rrf_weights=[0.0, 1.0],
    top_k=4,
)
pretty(f"dense only (dense={stats.dense_candidates} candidates)", results)

--- dense only (dense=4 candidates) ---
  [1] monitoring.md  score=0.032 via=fused — Watch the error rate and p99 latency dashboards for fifteen minutes after every ...
  [2] playbook.md    score=0.032 via=fused — When a deploy fails, follow the deploy playbook: page the on-call, start the rol...
  [3] rollback.md    score=0.032 via=fused — Always practice rolling back a deployment in staging before you actually need to...
  [4] migrations.md  score=0.016 via=dense — Run migrations in a dry-run mode first. Dry runs exercise the migration path wit...



## Variation 4 — Context window

Passing `context_window=N` asks memtomem to attach up to `N` adjacent
chunks from the same source file on either side of each hit. The
surrounding context is returned on the `context` attribute of each
`SearchResult`, which is `None` unless the window was requested.

In [8]:
results, _ = await comp.search_pipeline.search(
    query=query,
    context_window=2,
    top_k=2,
)

for r in results:
    name = Path(r.chunk.metadata.source_file).name
    print(f"[{r.rank}] {name} (score={r.score:.3f})")
    if r.context is None:
        print("  (no context expansion returned)\n")
        continue
    before = getattr(r.context, "before", []) or []
    after = getattr(r.context, "after", []) or []
    print(f"  context: {len(before)} chunk(s) before, {len(after)} after")
    for c in before:
        print(f"    - before: {c.content[:60].replace(chr(10), ' ')}...")
    for c in after:
        print(f"    - after : {c.content[:60].replace(chr(10), ' ')}...")
    print()

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=222399;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=418657;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

[1] monitoring.md (score=0.032)
  context: 0 chunk(s) before, 0 after

[2] playbook.md (score=0.032)
  context: 0 chunk(s) before, 0 after



## Config-time knobs: MMR, time decay, reranker

`rrf_weights` and `context_window` are the only two levers exposed as
call-time arguments. MMR diversification, time decay, and the cross-encoder
reranker are configured when `Components` is first created, so the recipe
to experiment with them is:

1. Build a `Mem2MemConfig()`.
2. Flip fields on `config.mmr`, `config.decay`, or `config.rerank`.
3. Call `create_components(config)` — this returns a fresh `Components`
   with the tweaked pipeline. You can keep multiple instances side by
   side and compare them.

```python
from memtomem.config import Mem2MemConfig

tuned_config = Mem2MemConfig()
tuned_config.storage.sqlite_path = db_path   # reuse the same DB
tuned_config.indexing.memory_dirs = [notes_dir]
tuned_config.mmr.enabled = True
tuned_config.mmr.lambda_param = 0.5          # 0 = max diversity, 1 = max relevance
tuned_config.decay.enabled = True
tuned_config.decay.half_life_days = 14.0

tuned = await create_components(tuned_config)
# ... search with tuned.search_pipeline ...
await close_components(tuned)
```

Because both `Components` share the same SQLite file, you do not need to
re-index for every tuning run — only for tokenizer changes (see notebook
02) and for schema changes.

## Cleanup

In [9]:
# Release connections and remove the temp directory.
await close_components(comp)
tmp.cleanup()

# Clean up the environment variables we set.
for key in (
    "MEMTOMEM_STORAGE__SQLITE_PATH",
    "MEMTOMEM_INDEXING__MEMORY_DIRS",
    "MEMTOMEM_EMBEDDING__PROVIDER",
    "MEMTOMEM_EMBEDDING__MODEL",
    "MEMTOMEM_EMBEDDING__DIMENSION",
):
    os.environ.pop(key, None)

print("clean.")

clean.
